## Additional

额外分析。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

corr_labels_cn = {
    'trial_calls': 'AI调用次数',
    'first_ai_time': '首次调用时间',
    'pre_first_call_ideas': '首次调用前想法数',
    'pre_think_time': '平均调用前思考时间',
    'perspective_taking': '观点采纳率',
    'affected_by_ai': '受AI影响率',
    'bfi_extroversion': '外向性',
    'bfi_agreeableness': '宜人性',
    'bfi_conscientiousness': '尽责性',
    'bfi_neuroticism': '神经质',
    'bfi_openness': '开放性',
    'cse_total': '创造性自我效能',
    'ribs_total': 'RIBS总分',
    'ai_attitude': 'AI态度',
    'dat_score': 'DAT分数',
    'originality': '原创性',
    'fluency': '流畅性',
    'above_median': '高质量回答数',
    'above_median_ratio': '高质量回答率',
    'cluster': '簇别',
    'age': '年龄',
    'gender': '性别',
}

# 输入文件路径
output_dir = Path('output')
trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'
assistant_originality_file = data_dir / 'scoring' / 'assistant_originality.csv'

### 大五人格特质调节分析

In [ ]:
# 加载数据
df_cluster = pd.read_csv(trial_with_cluster_file)
df_perf = pd.read_csv(performance_file)

# 以试次为单位合并聚类标签（保持原始 left join）
df_merged = pd.merge(
    df_perf,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
)

print(df_merged['cluster'].value_counts(dropna=False).sort_index())

# 准备调节分析数据
mod_df = df_merged[df_merged["cluster"] >= 0].copy()

def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

# 对所有参与分析的连续变量做 Z 标准化
for col in ["trial_calls", "first_ai_time", "pre_first_call_ideas", "pre_think_time",
            "cse_total", "ribs_total", "dat_score", "bfi_openness",
            "bfi_extroversion", "bfi_agreeableness", "bfi_conscientiousness",
            "bfi_neuroticism", "ai_attitude",
            "perspective_taking", "affected_by_ai", "age",
            "originality", "fluency"]:
    if col in mod_df.columns:
        mod_df[f"{col}_z"] = zscore(mod_df[col])

print(f"mod_df 样本量: {len(mod_df)}")

In [ ]:
# ---- 核心拟合函数 ----
# 与 03c_moderation.ipynb 保持一致

def _fit_mixed_or_ols(formula, data, has_groups):
    """尝试 LMM，失败则回退至 OLS。"""
    if has_groups:
        try:
            model = sm.MixedLM.from_formula(
                formula,
                groups="participant_id",
                vc_formula={"item_name": "0 + C(item_name)"},
                data=data
            )
            result = model.fit(reml=False, method="lbfgs", maxiter=2000)
            return result, "LMM"
        except Exception:
            pass
    model = ols(formula, data=data).fit()
    return model, "OLS"


def _fit_and_jn(predictor, moderator, outcome, data, controls=None, alpha=0.05, verbose=False):
    """拟合交互模型，返回 Johnson-Neyman 简单斜率数据。"""
    if controls is None:
        controls = []

    p_z = f"{predictor}_z" if f"{predictor}_z" in data.columns else predictor
    m_z = f"{moderator}_z" if f"{moderator}_z" in data.columns else moderator

    subset = data.dropna(subset=[p_z, m_z, outcome] + controls).copy()
    if len(subset) < 30:
        return None

    ctrl_str = " + ".join(controls) if controls else ""
    rhs = f"{p_z} * {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
    formula = f"{outcome} ~ {rhs}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    model, mtype = _fit_mixed_or_ols(formula, subset, has_groups)

    # 提取系数与协方差
    if mtype == "LMM":
        params = model.fe_params
        cov_fe = model.cov_params().loc[params.index, params.index]
        main_formula = f"{outcome} ~ {p_z} + {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
        try:
            model_main, _ = _fit_mixed_or_ols(main_formula, subset, has_groups)
            lr_stat = 2 * (model.llf - model_main.llf)
            p_inter = stats.chi2.sf(lr_stat, 1)
        except Exception:
            p_inter = np.nan
    else:
        params = model.params
        cov_fe = model.cov_params()
        if hasattr(cov_fe, "loc"):
            cov_fe = cov_fe.loc[params.index, params.index]
        main_formula = f"{outcome} ~ {p_z} + {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
        model_main = ols(main_formula, data=subset).fit()
        anova_res = sm.stats.anova_lm(model_main, model)
        p_inter = anova_res["Pr(>F)"].iloc[1] if "Pr(>F)" in anova_res.columns else np.nan

    # 定位交互项
    inter_terms = [k for k in params.index if ":" in k and p_z in k]
    if not inter_terms:
        inter_terms = [k for k in params.index if ":" in k]
    if not inter_terms:
        return None
    inter_key = inter_terms[0]

    b_pred = params.get(p_z)
    b_inter = params.get(inter_key)
    if b_pred is None or b_inter is None:
        return None

    try:
        v_pred = cov_fe.at[p_z, p_z]
        v_inter = cov_fe.at[inter_key, inter_key]
        cov_pi = cov_fe.at[p_z, inter_key]
    except KeyError:
        return None

    # ---- JN 计算 ----
    z_crit = stats.norm.ppf(1 - alpha / 2)

    m_raw = subset[moderator]
    m_model = subset[m_z]
    m_raw_min, m_raw_max = m_raw.min(), m_raw.max()
    m_model_min, m_model_max = m_model.min(), m_model.max()

    m_range = np.linspace(m_raw_min, m_raw_max, 300)
    m_model_range = np.linspace(m_model_min, m_model_max, 300)

    simple_slope = b_pred + b_inter * m_model_range
    var_slope = v_pred + m_model_range**2 * v_inter + 2 * m_model_range * cov_pi
    se_slope = np.sqrt(np.maximum(var_slope, 0))
    ci_lo = simple_slope - z_crit * se_slope
    ci_hi = simple_slope + z_crit * se_slope
    sig_mask = (ci_lo > 0) | (ci_hi < 0)

    # 解 JN 边界点
    A = b_inter**2 - z_crit**2 * v_inter
    B = 2 * (b_pred * b_inter - z_crit**2 * cov_pi)
    C = b_pred**2 - z_crit**2 * v_pred

    jn_points_raw = []
    if abs(A) > 1e-12:
        disc = B**2 - 4 * A * C
        if disc >= 0:
            sqrt_disc = np.sqrt(disc)
            for root_model in [(-B + sqrt_disc) / (2 * A), (-B - sqrt_disc) / (2 * A)]:
                if m_model_min <= root_model <= m_model_max:
                    root_raw = np.interp(root_model, [m_model_min, m_model_max], [m_raw_min, m_raw_max])
                    jn_points_raw.append(root_raw)

    if verbose:
        print(f"  [{predictor} x {moderator} -> {outcome}] {mtype}, p_inter={p_inter:.4f}")
        print(f"    斜率范围: [{simple_slope.min():.3f}, {simple_slope.max():.3f}]")
        print(f"    显著比例: {sig_mask.mean():.1%}")
        if jn_points_raw:
            print(f"    JN 边界: {jn_points_raw}")

    return {
        "m_range": m_range, "simple_slope": simple_slope,
        "ci_lo": ci_lo, "ci_hi": ci_hi, "sig_mask": sig_mask,
        "jn_points": sorted(jn_points_raw), "p_inter": p_inter,
        "p_cn": corr_labels_cn.get(predictor, predictor),
        "m_cn": corr_labels_cn.get(moderator, moderator),
        "y_cn": corr_labels_cn.get(outcome, outcome),
        "mtype": mtype, "m_raw": subset[moderator],
        "p_raw": subset[p_z], "y_raw": subset[outcome],
        "m_mean": m_raw.mean(), "m_sd": m_raw.std(ddof=0)
    }


def interaction_analysis(predictor, moderator, outcome, controls=None, do_jn=False, plot_cat_slopes=False):
    """详细的单次交互分析：模型拟合 + 可选 JN 图或分类简单斜率图。"""
    if controls is None:
        controls = ["age", "gender"]

    needed = [predictor, moderator, outcome] + controls
    missing = [c for c in needed if c not in mod_df.columns and f"{c}_z" not in mod_df.columns]
    if missing:
        print(f"跳过 {predictor}x{moderator}->{outcome}：缺少字段: {missing}")
        return

    p_z = f"{predictor}_z" if f"{predictor}_z" in mod_df.columns else predictor
    m_z = f"{moderator}_z" if f"{moderator}_z" in mod_df.columns else moderator
    y = outcome

    subset = mod_df.dropna(subset=[p_z, m_z, y] + controls).copy()
    if len(subset) < 30:
        print(f"样本量过小 ({len(subset)})，跳过 {predictor}x{moderator}->{outcome}")
        return

    if moderator == "cluster":
        subset[moderator] = subset[moderator].astype("category")

    ctrl_list = [c for c in controls if c in subset.columns]
    ctrl_str = " + ".join(ctrl_list) if ctrl_list else ""

    if ctrl_str:
        main_formula = f"{y} ~ {p_z} + {m_z} + {ctrl_str}"
        int_formula = f"{y} ~ {p_z} * {m_z} + {ctrl_str}"
    else:
        main_formula = f"{y} ~ {p_z} + {m_z}"
        int_formula = f"{y} ~ {p_z} * {m_z}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns

    model_main, mtype = _fit_mixed_or_ols(main_formula, subset, has_groups)
    model_int, _ = _fit_mixed_or_ols(int_formula, subset, has_groups)

    p_inter = None
    if mtype == "LMM":
        lr_stat = 2 * (model_int.llf - model_main.llf)
        df_main = len(model_main.fe_params)
        df_int = len(model_int.fe_params)
        df_diff = df_int - df_main
        if df_diff <= 0:
            return
        p_inter = stats.chi2.sf(lr_stat, df_diff)
        print(f"--- {y} ~ {p_z} x {m_z} (LMM, n={len(subset)}) ---")
        print(f"  LRT chi2({df_diff}) = {lr_stat:.2f}, p = {p_inter:.4f}")
        if p_inter < 0.05:
            print("  发现显著交互作用")
            print(model_int.summary().tables[1])
    else:
        anova_res = sm.stats.anova_lm(model_main, model_int)
        if "Pr(>F)" not in anova_res.columns or len(anova_res) <= 1:
            return
        p_inter = anova_res["Pr(>F)"].iloc[1]
        print(f"--- {y} ~ {p_z} x {m_z} (OLS, n={len(subset)}) ---")
        print(anova_res)
        if p_inter < 0.05:
            print("发现交互作用，完整模型结果:")
            print(model_int.summary().tables[1])

    # ---- Johnson-Neyman 简单斜率图 ----
    if do_jn and p_inter is not None and p_inter < 0.05:
        jn_res = _fit_and_jn(predictor, moderator, outcome, subset, controls=ctrl_list, verbose=True)
        if jn_res is None:
            return

        p_cn = jn_res["p_cn"]
        m_cn = jn_res["m_cn"]
        y_cn = jn_res["y_cn"]
        m_range = jn_res["m_range"]
        simple_slope = jn_res["simple_slope"]
        ci_lo = jn_res["ci_lo"]
        ci_hi = jn_res["ci_hi"]
        jn_points = jn_res["jn_points"]
        m_raw = jn_res["m_raw"]

        fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(8, 6),
            gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

        ax_top.plot(m_range, simple_slope, "b-", linewidth=2, label="简单斜率")
        ax_top.fill_between(m_range, ci_lo, ci_hi, alpha=0.15, color="b", label="95% CI")
        ax_top.axhline(y=0, color="gray", linestyle="--", linewidth=1)
        for jp in jn_points:
            ax_top.axvline(x=jp, color="#D4A017", linestyle="--", linewidth=1.2, alpha=0.9)
        ax_top.set_ylabel(f"{p_cn} 的简单斜率", fontsize=9)
        ax_top.legend(fontsize=8)
        sig_prop = ((ci_lo > 0) | (ci_hi < 0)).mean()
        p_str = f"p = {p_inter:.3f}" if p_inter >= 0.001 else "p < 0.001"
        ax_top.text(0.98, 0.03, f"{p_str} | 显著区间 {sig_prop:.0%}",
                    transform=ax_top.transAxes, ha="right", va="bottom",
                    fontsize=8, bbox=dict(boxstyle="round,pad=0.2", facecolor="wheat", alpha=0.7))
        ax_top.set_title(f"{p_cn} × {m_cn} → {y_cn}", fontsize=11, fontweight="bold")

        ax_bot.hist(m_raw, bins=40, color="lightgray", edgecolor="white")
        ax_bot.set_xlabel(m_cn, fontsize=10)
        ax_bot.set_ylabel("频数", fontsize=8)
        fig.tight_layout()
        plt.show()

    # ---- 分类调节变量简单斜率图 ----
    if plot_cat_slopes and p_inter is not None and p_inter < 0.05:
        simple_slopes_categorical(predictor, moderator, outcome, data=subset, controls=ctrl_list)


def simple_slopes_categorical(predictor, moderator, outcome, data=None, controls=None, alpha=0.05):
    """分类调节变量的简单斜率估计与可视化。"""
    if data is None:
        data = mod_df
    if controls is None:
        controls = ["age", "gender"]

    p_cn = corr_labels_cn.get(predictor, predictor)
    m_cn = corr_labels_cn.get(moderator, moderator)
    y_cn = corr_labels_cn.get(outcome, outcome)

    needed = [predictor, moderator, outcome] + list(controls)
    missing = [c for c in needed if c not in data.columns]
    if missing:
        print(f"跳过 {p_cn} x {m_cn} -> {y_cn}：缺少字段: {missing}")
        return None, None

    subset = data.dropna(subset=needed).copy()
    if len(subset) < 30:
        print(f"样本量过小 ({len(subset)})，跳过 {p_cn} x {m_cn} -> {y_cn}")
        return None, None

    subset[moderator] = subset[moderator].astype("category")

    x_term = f"{predictor}_z" if f"{predictor}_z" in subset.columns else predictor

    control_terms = []
    for c in controls:
        if c in (predictor, moderator, outcome):
            continue
        if c not in subset.columns:
            continue
        if (isinstance(subset[c].dtype, pd.CategoricalDtype)
                or subset[c].dtype == object
                or pd.api.types.is_bool_dtype(subset[c])):
            control_terms.append(f"C({c})")
        else:
            control_terms.append(c)

    full_formula = f"{outcome} ~ {x_term} * C({moderator})"
    if control_terms:
        full_formula += " + " + " + ".join(control_terms)

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    full_model, mtype = _fit_mixed_or_ols(full_formula, subset, has_groups)

    print(f"--- {y_cn} ~ {p_cn} x {m_cn} 简单斜率 ({mtype}) ---")

    if mtype == "LMM":
        params = full_model.fe_params
        cov = pd.DataFrame(
            full_model.cov_params(),
            index=full_model.fe_params.index,
            columns=full_model.fe_params.index
        )
    else:
        params = full_model.params
        cov_raw = full_model.cov_params()
        if hasattr(cov_raw, "loc"):
            cov = cov_raw.loc[params.index, params.index]
        else:
            cov = pd.DataFrame(cov_raw, index=params.index, columns=params.index)

    levels = list(subset[moderator].cat.categories)
    ref_level = levels[0]

    def _slope_contrast(level):
        contrast = pd.Series(0.0, index=params.index)
        if x_term not in contrast.index:
            return None
        contrast[x_term] = 1.0
        if level != ref_level:
            level_tag = f"[T.{level}]"
            candidates = [n for n in params.index
                          if x_term in n and level_tag in n]
            if not candidates:
                return None
            contrast[candidates[0]] = 1.0
        return contrast

    slope_rows = []
    slope_vectors = {}
    df_resid = getattr(full_model, "df_resid", max(1, len(subset) - len(params)))
    tcrit = stats.t.ppf(1 - alpha / 2, df_resid)

    for level in levels:
        L = _slope_contrast(level)
        if L is None:
            print(f"  无法识别水平 {level} 的简单斜率，跳过")
            continue

        slope = float(L @ params)
        var = float(L @ cov.values @ L.values)
        se = np.sqrt(var) if var >= 0 else np.nan
        tval = slope / se if (se and not np.isnan(se)) else np.nan
        pval = 2 * stats.t.sf(abs(tval), df_resid) if not np.isnan(tval) else np.nan

        ci_lo = slope - tcrit * se if not np.isnan(se) else np.nan
        ci_hi = slope + tcrit * se if not np.isnan(se) else np.nan

        slope_rows.append({
            "level": level, "slope": slope, "se": se,
            "t": tval, "p": pval, "ci_lo": ci_lo, "ci_hi": ci_hi
        })
        slope_vectors[level] = L

    slope_df = pd.DataFrame(slope_rows)
    if len(slope_df):
        slope_df["p_bonf"] = multipletests(slope_df["p"].values, method="bonferroni")[1]
        slope_df["p_fdr"] = multipletests(slope_df["p"].values, method="fdr_bh")[1]

    print(f"--- {y_cn} 在 {m_cn} 各水平下的简单斜率 ---")
    print(slope_df.round(4))

    # 成对斜率比较
    pair_rows = []
    if len(slope_vectors) >= 2:
        for a, b in combinations(levels, 2):
            if a not in slope_vectors or b not in slope_vectors:
                continue
            D = slope_vectors[a] - slope_vectors[b]
            diff = float(D @ params)
            var = float(D @ cov.values @ D.values)
            se = np.sqrt(var) if var >= 0 else np.nan
            tval = diff / se if (se and not np.isnan(se)) else np.nan
            pval = 2 * stats.t.sf(abs(tval), df_resid) if not np.isnan(tval) else np.nan
            ci_lo = diff - tcrit * se if not np.isnan(se) else np.nan
            ci_hi = diff + tcrit * se if not np.isnan(se) else np.nan
            pair_rows.append({
                "level1": a, "level2": b, "slope_diff": diff,
                "se": se, "t": tval, "p": pval,
                "ci_lo": ci_lo, "ci_hi": ci_hi
            })

    pair_df = pd.DataFrame(pair_rows)
    if len(pair_df):
        pair_df["p_bonf"] = multipletests(pair_df["p"].values, method="bonferroni")[1]
        pair_df["p_fdr"] = multipletests(pair_df["p"].values, method="fdr_bh")[1]
        print(f"--- {y_cn} 的简单斜率组间比较 ---")
        print(pair_df.round(4))

    # ---- 交互效应图 ----
    fig, ax = plt.subplots(figsize=(6, 5))

    x_vals = subset[x_term]
    x_range = np.linspace(x_vals.min(), x_vals.max(), 100)

    if moderator == "cluster":
        level_labels = {lvl: cluster_name_map.get(int(lvl), str(int(lvl))) for lvl in levels}
    else:
        level_labels = {lvl: corr_labels_cn.get(lvl, str(lvl)) for lvl in levels}

    colors = plt.cm.Set2(np.linspace(0, 1, len(levels)))

    for i, level in enumerate(levels):
        intercept = params.get("Intercept", 0.0)
        if level != ref_level:
            for key in params.index:
                if f"C({moderator})[T.{level}]" in key and ":" not in key:
                    intercept += params[key]
                    break

        slope_val = slope_df.loc[slope_df["level"] == level, "slope"].values[0]
        y_pred = intercept + slope_val * x_range

        ax.plot(x_range, y_pred, color=colors[i], linewidth=2.5,
                label=level_labels[level])

        mask = subset[moderator] == level
        ax.scatter(subset.loc[mask, x_term], subset.loc[mask, outcome],
                   color=colors[i], alpha=0.12, s=8, edgecolors="none")

    ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
    ax.legend(fontsize=9, title=m_cn)
    ax.set_xlabel(p_cn)
    ax.set_ylabel(y_cn)
    ax.set_title(f"{p_cn} x {m_cn} → {y_cn}")
    fig.tight_layout()
    plt.show()

    return slope_df, pair_df

In [ ]:
# 大五人格特质作为调节变量
personality_traits = ['bfi_extroversion', 'bfi_agreeableness', 'bfi_conscientiousness', 'bfi_neuroticism', 'bfi_openness']

for trait in personality_traits:
    print(f"\n{'='*60}")
    print(f"调节变量: {corr_labels_cn.get(trait, trait)}")
    print(f"{'='*60}")

    # 原创性
    interaction_analysis('trial_calls', trait, 'originality', do_jn=True)
    interaction_analysis('first_ai_time', trait, 'originality', do_jn=True)
    interaction_analysis('pre_first_call_ideas', trait, 'originality', do_jn=True)
    interaction_analysis('pre_think_time', trait, 'originality', do_jn=True)
    interaction_analysis('perspective_taking', trait, 'originality', do_jn=True)
    interaction_analysis('affected_by_ai', trait, 'originality', do_jn=True)

    # 流畅性
    interaction_analysis('trial_calls', trait, 'fluency', do_jn=True)
    interaction_analysis('first_ai_time', trait, 'fluency', do_jn=True)
    interaction_analysis('pre_first_call_ideas', trait, 'fluency', do_jn=True)
    interaction_analysis('pre_think_time', trait, 'fluency', do_jn=True)
    interaction_analysis('perspective_taking', trait, 'fluency', do_jn=True)
    interaction_analysis('affected_by_ai', trait, 'fluency', do_jn=True)

    # 高质量回答数
    interaction_analysis('trial_calls', trait, 'above_median', do_jn=True)
    interaction_analysis('first_ai_time', trait, 'above_median', do_jn=True)
    interaction_analysis('pre_first_call_ideas', trait, 'above_median', do_jn=True)
    interaction_analysis('pre_think_time', trait, 'above_median', do_jn=True)
    interaction_analysis('perspective_taking', trait, 'above_median', do_jn=True)
    interaction_analysis('affected_by_ai', trait, 'above_median', do_jn=True)

    # 高质量回答比率
    interaction_analysis('trial_calls', trait, 'above_median_ratio', do_jn=True)
    interaction_analysis('first_ai_time', trait, 'above_median_ratio', do_jn=True)
    interaction_analysis('pre_first_call_ideas', trait, 'above_median_ratio', do_jn=True)
    interaction_analysis('pre_think_time', trait, 'above_median_ratio', do_jn=True)
    interaction_analysis('perspective_taking', trait, 'above_median_ratio', do_jn=True)
    interaction_analysis('affected_by_ai', trait, 'above_median_ratio', do_jn=True)

In [ ]:
# ---- 汇总 JN 网格图函数 ----
def plot_moderation_grid_jn(interactions, data, n_cols=3, controls=None, figsize_per_cell=(4.5, 3.5)):
    """连续调节变量 JN 网格汇总图。"""
    if controls is None:
        controls = ["age", "gender"]

    valid = []
    for pred, mod, out in interactions:
        res = _fit_and_jn(pred, mod, out, data, controls)
        if res is not None:
            valid.append(res)

    n_rows = (len(valid) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_cell[0] * n_cols, figsize_per_cell[1] * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, res in enumerate(valid):
        ax = axes[i]

        if res["sig_mask"].any():
            in_sig = False; start = None
            for j, sig in enumerate(res["sig_mask"]):
                if sig and not in_sig:
                    start = res["m_range"][j]
                    in_sig = True
                elif not sig and in_sig:
                    ax.axvspan(start, res["m_range"][j], color="#FFF3B0", alpha=0.6, lw=0)
                    in_sig = False
            if in_sig:
                ax.axvspan(start, res["m_range"][-1], color="#FFF3B0", alpha=0.6, lw=0)

        ax.fill_between(res["m_range"], res["ci_lo"], res["ci_hi"], color="#2c7bb6", alpha=0.10, lw=0)
        ax.plot(res["m_range"], res["simple_slope"], color="#2c7bb6", linewidth=2)
        ax.axhline(y=0, color="gray", linestyle=":", linewidth=0.8)

        for jp in res["jn_points"]:
            ax.axvline(x=jp, color="#D4A017", linestyle="--", linewidth=1, alpha=0.8)

        ax.set_xlabel(res["m_cn"], fontsize=9)
        ax.set_ylabel(f'{res["p_cn"]} 的简单斜率', fontsize=9)
        ax.set_title(f'{res["p_cn"]} x {res["m_cn"]} → {res["y_cn"]}', fontsize=10, fontweight="bold")

    for j in range(len(valid), len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()
    return fig

In [ ]:
# ===== 大五人格（开放性）JN 汇总图 =====
sig_bfi_openness = [
    ("trial_calls",  "bfi_openness", "originality"),
    ("first_ai_time",  "bfi_openness", "fluency"),
    ("pre_think_time", "bfi_openness", "above_median"),
]

print("开始生成 BFI-开放性 JN 汇总图...")
fig = plot_moderation_grid_jn(sig_bfi_openness, mod_df, n_cols=3)
fig.savefig("output/moderation_bfi_openness_jn.png", dpi=200, bbox_inches="tight")
plt.show()

### 聚类模式差异分析

高效采纳与审慎批判之间的区别？为什么采纳率会低？是因为个体特质？还是AI提示质量有差别？

In [ ]:
# 加载AI回答原创性评分
df_assistant_orig = pd.read_csv(assistant_originality_file, encoding='utf-8-sig')
print(f"AI回答评分数据: {len(df_assistant_orig)} 条回答")
print(f"缺失原创性评分: {df_assistant_orig['originality'].isna().sum()} 条")

# 以试次为单位聚合AI回答的原创性（均值），保留 item_name 用于随机效应
ai_orig_trial = df_assistant_orig.groupby(['participant_id', 'trial_index', 'item_name'], as_index=False)['originality'].mean()
ai_orig_trial.rename(columns={'originality': 'ai_originality_mean'}, inplace=True)

# 合并聚类标签与个体特质协变量（从已加载的 df_cluster 获取）
covariate_cols = ['cluster', 'dat_score', 'ribs_total', 'cse_total', 'ai_attitude',
                  'bfi_extroversion', 'bfi_agreeableness', 'bfi_conscientiousness',
                  'bfi_neuroticism', 'bfi_openness', 'age', 'gender']
cov_df = df_cluster[['participant_id', 'trial_index'] + [c for c in covariate_cols if c in df_cluster.columns]].copy()

df_merge = pd.merge(ai_orig_trial, cov_df, on=['participant_id', 'trial_index'], how='inner')

# 排除无调用试次（cluster == -1）
lmm_ai_df = df_merge[df_merge['cluster'] >= 0].copy()
lmm_ai_df = lmm_ai_df.dropna(subset=['ai_originality_mean', 'cluster', 'dat_score', 'ribs_total', 'cse_total', 'ai_attitude', 'participant_id', 'item_name'])

# 设置类型
lmm_ai_df['cluster'] = lmm_ai_df['cluster'].astype('category')
lmm_ai_df['participant_id'] = lmm_ai_df['participant_id'].astype('category')
lmm_ai_df['item_name'] = lmm_ai_df['item_name'].astype('category')
lmm_ai_df['cluster_name'] = lmm_ai_df['cluster'].map(cluster_name_map)

print(f"\nLMM 分析数据规模: {lmm_ai_df.shape}")
print(f"被试数: {lmm_ai_df['participant_id'].nunique()}")
print(f"物品数: {lmm_ai_df['item_name'].nunique()}")
print(f"\ncluster 分布:\n{lmm_ai_df['cluster'].value_counts().sort_index()}")
print(f"\n各聚类AI回答原创性均值:\n{lmm_ai_df.groupby('cluster_name')['ai_originality_mean'].describe().round(3)}")


In [ ]:
# ---- LMM - 交叉随机截距模型：比较三个 cluster 在 AI 回答原创性上的差异 ----
# 固定效应：cluster + 个体特质协变量
# 随机效应：participant_id（被试随机截距）+ item_name（题目随机截距）

lmm_ai = sm.MixedLM.from_formula(
    formula='ai_originality_mean ~ C(cluster) + dat_score + ribs_total + cse_total + ai_attitude',
    groups='participant_id',
    re_formula='1',
    vc_formula={'item_name': '0 + C(item_name)'},
    data=lmm_ai_df
)

lmm_ai_res = lmm_ai.fit(reml=False, method='lbfgs', maxiter=2000)
print(lmm_ai_res.summary())

# 固定效应项的 Wald 检验
print('\n固定效应 Wald 检验:')
print(lmm_ai_res.wald_test_terms(skip_single=False, scalar=True))

# 估计边际均值（协变量固定为样本均值，随机效应 = 0）
pred_df_ai = pd.DataFrame({'cluster': lmm_ai_df['cluster'].cat.categories})
pred_df_ai['participant_id'] = '__marginal__'
pred_df_ai['item_name'] = '__marginal__'
for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
    pred_df_ai[cov] = lmm_ai_df[cov].mean()
pred_df_ai['pred_ai_originality'] = lmm_ai_res.predict(pred_df_ai)
pred_df_ai['cluster_name'] = pred_df_ai['cluster'].map(cluster_name_map)
print('\n各 cluster 的预测 AI 回答原创性（边际均值）:')
display(pred_df_ai[['cluster', 'cluster_name', 'pred_ai_originality']])


In [ ]:
# ---- LMM 基于固定效应的两两比较（基于估计边际均值 / LSM）----
# 原理：使用固定效应参数与其协方差矩阵，构建各 cluster 的设计向量，
# 计算组间差值的估计值、标准误、t 统计量与 p 值（并提供置信区间），
# 最后做多重比较校正（Bonferroni 和 FDR）。
import patsy
from itertools import combinations
from scipy import stats
from statsmodels.stats.multitest import multipletests

# 提取固定效应参数与协方差
fe_params_ai = lmm_ai_res.fe_params
cov_fe_ai = lmm_ai_res.cov_params()
if hasattr(cov_fe_ai, 'loc'):
    cov_fe_ai = cov_fe_ai.loc[fe_params_ai.index, fe_params_ai.index]
else:
    cov_fe_ai = pd.DataFrame(cov_fe_ai, index=fe_params_ai.index, columns=fe_params_ai.index)

# 构建设计矩阵
clusters_ai = lmm_ai_df['cluster'].cat.categories
pred_design_ai = pd.DataFrame({'cluster': clusters_ai})
for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
    if cov in lmm_ai_df.columns:
        pred_design_ai[cov] = lmm_ai_df[cov].mean()

design_ai = patsy.dmatrix('C(cluster) + dat_score + ribs_total + cse_total + ai_attitude',
                          pred_design_ai, return_type='dataframe')
for col in fe_params_ai.index:
    if col not in design_ai.columns:
        design_ai[col] = 0.0
design_ai = design_ai[fe_params_ai.index]

pred_means_ai = design_ai.dot(fe_params_ai)
df_resid_ai = getattr(lmm_ai_res, 'df_resid', max(1, lmm_ai_df.shape[0] - len(fe_params_ai)))

pairwise_results_ai = []
for i, j in combinations(range(len(clusters_ai)), 2):
    xa = design_ai.iloc[i].values
    xb = design_ai.iloc[j].values
    contrast = xa - xb
    diff = float(contrast.dot(fe_params_ai.values))
    var = float(contrast.dot(cov_fe_ai.values).dot(contrast))
    se = (var ** 0.5) if var >= 0 else float('nan')
    tstat = diff / se if se != 0 else float('nan')
    pval = 2 * stats.t.sf(abs(tstat), df_resid_ai) if not np.isnan(tstat) else float('nan')
    tcrit = stats.t.ppf(0.975, df_resid_ai)
    ci_lo = diff - tcrit * se
    ci_hi = diff + tcrit * se
    name1 = cluster_name_map[int(clusters_ai[i])]
    name2 = cluster_name_map[int(clusters_ai[j])]
    pairwise_results_ai.append({
        'group1': name1,
        'group2': name2,
        'mean1': float(pred_means_ai.iloc[i]),
        'mean2': float(pred_means_ai.iloc[j]),
        'mean_diff': diff,
        'se': se,
        't': tstat,
        'p': pval,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi
    })

pairwise_ai_df = pd.DataFrame(pairwise_results_ai).sort_values('p')
if len(pairwise_ai_df):
    pairwise_ai_df['p_bonf'] = multipletests(pairwise_ai_df['p'].values, method='bonferroni')[1]
    pairwise_ai_df['p_fdr'] = multipletests(pairwise_ai_df['p'].values, method='fdr_bh')[1]

print('基于 LMM 固定效应的两两比较（按 p 值排序）:')
display(pairwise_ai_df[['group1', 'group2', 'mean1', 'mean2', 'mean_diff', 'se', 't', 'p', 'p_bonf', 'p_fdr', 'ci_lo', 'ci_hi']].round(4))


In [ ]:
# ---- 可视化 ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 箱线图（原始数据）
ax = axes[0]
order = sorted(lmm_ai_df['cluster'].cat.categories)
colors_box = ['#FF9999', '#66B2FF', '#99FF99']
bp = ax.boxplot([lmm_ai_df.loc[lmm_ai_df['cluster'] == c, 'ai_originality_mean'] for c in order],
                labels=[cluster_name_map[int(c)] for c in order],
                patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
for i, c in enumerate(order):
    vals = lmm_ai_df.loc[lmm_ai_df['cluster'] == c, 'ai_originality_mean']
    strip_x = np.random.normal(i+1, 0.04, size=len(vals))
    ax.scatter(strip_x, vals, alpha=0.3, s=8, color=colors_box[i])
ax.set_ylabel('AI回答原创性均值')
ax.set_title('各聚类AI回答原创性分布（原始数据）')

# LMM 估计边际均值条形图 + 误差线
ax = axes[1]
x_pos = np.arange(len(order))
means_pred = pred_df_ai['pred_ai_originality'].values
# 用预测值的 SE 近似（从两两比较表中提取每组 SE）
se_pred = []
for c in order:
    # 对每个水平，从 pairwise 表中找对应 SE
    rows = pairwise_ai_df[(pairwise_ai_df['group1'] == cluster_name_map[int(c)]) | (pairwise_ai_df['group2'] == cluster_name_map[int(c)])]
    if len(rows) > 0:
        se_pred.append(rows['se'].mean())  # 平均 SE 作为近似
    else:
        se_pred.append(0)
ax.bar(x_pos, means_pred, yerr=se_pred, capsize=5,
       color=colors_box, alpha=0.7, width=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([cluster_name_map[int(c)] for c in order])
ax.set_ylabel('AI回答原创性（LMM估计边际均值）')
ax.set_title('各聚类AI回答原创性LMM估计边际均值')

fig.tight_layout()
plt.show()
